In [1]:
import kagglehub
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
import pickle

In [2]:
path = kagglehub.dataset_download("kartik2112/fraud-detection")
print("Path:", path)
for f in os.listdir(path):
    size = os.path.getsize(os.path.join(path, f)) / 1e6
    print(f"  {f}  ({size:.1f} MB)")

100%|██████████| 202M/202M [00:05<00:00, 38.6MB/s]

Extracting files...


Path: /root/.cache/kagglehub/datasets/kartik2112/fraud-detection/versions/1
  fraudTrain.csv  (351.2 MB)
  fraudTest.csv  (150.4 MB)


In [3]:

# ----------Load the files--------------------------
print("Loading files")
train_df = pd.read_csv(f"{path}/fraudTrain.csv")
test_df  = pd.read_csv(f"{path}/fraudTest.csv")

print(f"Train : {train_df.shape}")
print(f"Test  : {test_df.shape}")
print(f"\nColumns: {train_df.columns.tolist()}")
print(f"\nSample:")
print(train_df.head(2))

# -----------------Class balance -----------------------------------------
print(f"\nTrain Fraud : {train_df['is_fraud'].sum():,}  ({train_df['is_fraud'].mean()*100:.2f}%)")
print(f"Train Legit : {(train_df['is_fraud']==0).sum():,}")
print(f"Test  Fraud : {test_df['is_fraud'].sum():,}  ({test_df['is_fraud'].mean()*100:.2f}%)")
print(f"Test  Legit : {(test_df['is_fraud']==0).sum():,}")

Loading files
Train : (1296675, 23)
Test  : (555719, 23)

Columns: ['Unnamed: 0', 'trans_date_trans_time', 'cc_num', 'merchant', 'category', 'amt', 'first', 'last', 'gender', 'street', 'city', 'state', 'zip', 'lat', 'long', 'city_pop', 'job', 'dob', 'trans_num', 'unix_time', 'merch_lat', 'merch_long', 'is_fraud']

Sample:
   Unnamed: 0 trans_date_trans_time            cc_num  \
0           0   2019-01-01 00:00:18  2703186189652095   
1           1   2019-01-01 00:00:44      630423337322   

                          merchant     category     amt      first   last  \
0       fraud_Rippin, Kub and Mann     misc_net    4.97   Jennifer  Banks   
1  fraud_Heller, Gutmann and Zieme  grocery_pos  107.23  Stephanie   Gill   

  gender                        street  ...      lat      long  city_pop  \
0      F                561 Perry Cove  ...  36.0788  -81.1781      3495   
1      F  43039 Riley Greens Suite 393  ...  48.8878 -118.2105       149   

                                 job       

In [4]:
train_df["split"] = "train"
test_df["split"]  = "test"
df = pd.concat([train_df, test_df], axis=0).reset_index(drop=True)
print(f"Combined : {df.shape}")

# -------------- Parse datetime -----------------------------
df["trans_datetime"] = pd.to_datetime(df["trans_date_trans_time"])
df = df.sort_values(["cc_num", "trans_datetime"]).reset_index(drop=True)

# ------------------ Save text context for LLM ------------------------------------
text_cols = ["merchant", "category", "amt", "first", "last",
             "gender", "city", "state", "lat", "long",
             "merch_lat", "merch_long", "city_pop", "job", "is_fraud"]
df_text = df[["trans_num", "trans_date_trans_time", "cc_num"] + text_cols].copy()
os.makedirs("/content/fraud_data", exist_ok=True)
df_text.to_csv("/content/fraud_data/transaction_context.csv", index=False)
print(f"Text context saved - {df_text.shape}")

# FAST FEATURE ENGINEERING- Pandas rolling

print("\nEngineering features")

# ------------------------ 1. Time features ----------------------------
df["hour"]          = df["trans_datetime"].dt.hour
df["day_of_week"]   = df["trans_datetime"].dt.dayofweek
df["is_weekend"]    = (df["day_of_week"] >= 5).astype(int)
df["is_night"]      = ((df["hour"] >= 22) | (df["hour"] <= 6)).astype(int)
df["is_late_night"] = ((df["hour"] >= 2)  & (df["hour"] <= 6)).astype(int)
print("  Time features done")

# ------------------------------- 2. Distance -------------------------------------------
def haversine(lat1, lon1, lat2, lon2):
    R    = 3958.8
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a= np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

df["merch_distance"] = haversine(
    df["lat"].values, df["long"].values,
    df["merch_lat"].values, df["merch_long"].values
)
print("Distance done")

# -------------------- 3. Age----------------------------------------------
df["dob"] = pd.to_datetime(df["dob"])
df["age"] = (df["trans_datetime"] - df["dob"]).dt.days // 365
print("  Age done")

#-----------------------4.Set datetime as index ---------------------------------
df = df.set_index("trans_datetime")
# Group by cardholder
grp = df.groupby("cc_num")["amt"]

df["tx_count_1hr"] = grp.transform(lambda x: x.rolling("1h",  closed="left").count())
df["tx_count_1d"]  = grp.transform(lambda x: x.rolling("1D",  closed="left").count())
df["tx_count_7d"]  = grp.transform(lambda x: x.rolling("7D",  closed="left").count())
df["tx_count_30d"] = grp.transform(lambda x: x.rolling("30D", closed="left").count())


# Averages
df["avg_amt_1hr"] = grp.transform(lambda x: x.rolling("1h",  closed="left").mean()).fillna(df["amt"])
df["avg_amt_1d"]  = grp.transform(lambda x: x.rolling("1D",  closed="left").mean()).fillna(df["amt"])
df["avg_amt_7d"]  = grp.transform(lambda x: x.rolling("7D",  closed="left").mean()).fillna(df["amt"])
df["avg_amt_30d"] = grp.transform(lambda x: x.rolling("30D", closed="left").mean()).fillna(df["amt"])

# Reset index
df = df.reset_index()

#-------------- 5. Amount ratios ---------------------------------------
df["amt_ratio_1d"]  = df["amt"] / (df["avg_amt_1d"]  + 1e-6)
df["amt_ratio_7d"]  = df["amt"] / (df["avg_amt_7d"]  + 1e-6)
df["amt_ratio_30d"] = df["amt"] / (df["avg_amt_30d"] + 1e-6)

# ------------------- 6. Weekend ratio ----------------------------------------
df["weekend_ratio"] = df.groupby("cc_num")["is_weekend"].transform("mean")
print("  Amount ratios + weekend ratio done")


#-------------------------- ENCODE + DROP Columns--------------------------------

print("\nEncoding categoricals")
cat_cols = ["merchant", "category", "gender", "state", "job"]
le = LabelEncoder()
for col in cat_cols:
    df[col] = le.fit_transform(df[col].astype(str))

drop_cols = [
    "Unnamed: 0", "trans_date_trans_time", "trans_datetime",
    "cc_num", "first", "last", "street", "city",
    "zip", "dob", "trans_num", "split"
]
df.drop(columns=drop_cols, inplace=True)
print(f"Features after engineering : {df.shape[1]-1}")

#---------------------- SPLIT + SCALE + SAVE---------------------------

train = df.iloc[:len(train_df)].copy()
test  = df.iloc[len(train_df):].copy()

X_train = train.drop(columns=["is_fraud"]).values.astype(np.float32)
y_train = train["is_fraud"].values
X_test  = test.drop(columns=["is_fraud"]).values.astype(np.float32)
y_test  = test["is_fraud"].values


scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

X_legit_ae  = X_train[y_train == 0]
X_all_ae    = np.vstack([X_train, X_test])
y_all       = np.concatenate([y_train, y_test])
X_train_xgb = X_train
X_test_xgb  = X_test

np.save("/content/fraud_data/X_legit_ae.npy",  X_legit_ae)
np.save("/content/fraud_data/X_all_ae.npy",    X_all_ae)
np.save("/content/fraud_data/y_all.npy",        y_all)
np.save("/content/fraud_data/X_train_xgb.npy", X_train_xgb)
np.save("/content/fraud_data/X_test_xgb.npy",  X_test_xgb)
np.save("/content/fraud_data/y_train.npy",      y_train)
np.save("/content/fraud_data/y_test.npy",       y_test)

with open("/content/fraud_data/scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

print(" Done!")
print(f"  input_dim   : {X_legit_ae.shape[1]}")
print(f"  Fraud rate  : {y_all.mean()*100:.2f}%")
print(f"  Legit rows  : {(y_all==0).sum():,}")
print(f"  Fraud rows  : {(y_all==1).sum():,}")

Combined : (1852394, 24)
Text context saved - (1852394, 18)

Engineering features
  Time features done
Distance done
  Age done
  Amount ratios + weekend ratio done

Encoding categoricals
Features after engineering : 31
 Done!
  input_dim   : 31
  Fraud rate  : 0.52%
  Legit rows  : 1,842,743
  Fraud rows  : 9,651


In [5]:
#-------------- AUTOENCODER - Hyperparameter Tuning + Best Config Metrics--------------
import time
import os
import warnings
import numpy as np
import torch
import torch.nn as nn
from torch.amp import autocast, GradScaler
from sklearn.metrics import classification_report, roc_auc_score

warnings.filterwarnings("ignore", category=UserWarning)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_float32_matmul_precision('high')
torch._dynamo.config.recompile_limit = 64

SAVE_DIR = "/content/fraud_data/models"
os.makedirs(SAVE_DIR, exist_ok=True)

# ------------------------------------------ Load data----------------------------------------------------------
X_legit_ae = np.load("/content/fraud_data/X_legit_ae.npy")
X_all_ae   = np.load("/content/fraud_data/X_all_ae.npy")
y_all      = np.load("/content/fraud_data/y_all.npy")

# ------------------------ Clean NaN/inf ----------------------------------------------
print(f"NaN in X_legit_ae : {np.isnan(X_legit_ae).sum()}")
print(f"NaN in X_all_ae   : {np.isnan(X_all_ae).sum()}")
X_legit_ae = np.nan_to_num(X_legit_ae, nan=0.0, posinf=0.0, neginf=0.0)
X_all_ae   = np.nan_to_num(X_all_ae,   nan=0.0, posinf=0.0, neginf=0.0)
print("NaN/inf cleaned")

#---------------------- MODEL-----------------------------------------

def build_ae(input_dim, hidden, bottleneck):
    return nn.Sequential(
        nn.Linear(input_dim, hidden), nn.ReLU(),
        nn.Linear(hidden, bottleneck), nn.ReLU(),
        nn.Linear(bottleneck, hidden), nn.ReLU(),
        nn.Linear(hidden, input_dim),
    )

# -------------------------------------Training--------------------------------------

def train_ae(model, X_gpu, optimizer, loss_fn, epochs,
             scaler, batch_size, scheduler=None):
    model.train()
    n     = X_gpu.shape[0]
    steps = max(n // batch_size, 1)
    for epoch in range(epochs):
        total = 0
        idx   = torch.randperm(n, device=X_gpu.device)
        for start in range(0, n, batch_size):
            batch = X_gpu[idx[start:start + batch_size]]
            optimizer.zero_grad(set_to_none=True)
            with autocast("cuda"):
                loss = loss_fn(model(batch), batch)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            total += loss.item()
        if scheduler:
            scheduler.step()
        if (epoch + 1) % 50 == 0:
            print(f"    Epoch [{epoch+1}/{epochs}]  "
                  f"Loss: {total/steps:.6f}  "
                  f"LR: {optimizer.param_groups[0]['lr']:.7f}")
    return total / steps

#---------------------- Quick Evaluation on F1 only, used during tuning loop--------------------


def quick_eval(model, X_tensor, y_all):
    model.eval()
    with torch.no_grad():
        errors = torch.mean(
            (X_tensor - model(X_tensor)) ** 2, dim=1
        ).cpu().numpy()

    threshold = np.percentile(errors[y_all == 0], 95)
    preds     = (errors > threshold).astype(int)

    tp = ((preds == 1) & (y_all == 1)).sum()
    fp = ((preds == 1) & (y_all == 0)).sum()
    fn = ((preds == 0) & (y_all == 1)).sum()

    precision = tp / (tp + fp + 1e-9)
    recall    = tp / (tp + fn + 1e-9)
    f1        = 2 * precision * recall / (precision + recall + 1e-9)

    return f1, threshold, errors


#------------------------- FULL METRICS  onlyon best model-------------------


def full_metrics(model, X_tensor, y_all, threshold):
    model.eval()
    with torch.no_grad():
        errors = torch.mean(
            (X_tensor - model(X_tensor)) ** 2, dim=1
        ).cpu().numpy()

    preds = (errors > threshold).astype(int)

    tp = ((preds == 1) & (y_all == 1)).sum()
    fp = ((preds == 1) & (y_all == 0)).sum()
    fn = ((preds == 0) & (y_all == 1)).sum()
    tn = ((preds == 0) & (y_all == 0)).sum()

    recall    = tp / (tp + fn + 1e-9) * 100
    precision = tp / (tp + fp + 1e-9) * 100
    fpr       = fp / (fp + tn + 1e-9) * 100
    f1        = 2 * precision * recall / (precision + recall + 1e-9)
    accuracy  = (tp + tn) / len(y_all) * 100
    auc       = roc_auc_score(y_all, errors)

    print("\nBEST MODEL - PERFORMANCE METRICS")
    print(f"  Threshold  : {threshold:.6f}  (95th percentile)")
    print(f"  Accuracy   : {accuracy:.2f}%")
    print(f"  Recall     : {recall:.2f}%")
    print(f"  Precision  : {precision:.2f}%")
    print(f"  F1 Score   : {f1:.4f}")
    print(f"  FPR        : {fpr:.2f}%")
    print(f"  ROC-AUC    : {auc:.4f}")
    print("\nClassification Report:")
    print(classification_report(
        y_all, preds,
        target_names=["Legit", "Fraud"],
        digits=4))


#--------------- Save the best model----------------------


def save_model(model, cfg, f1, threshold, input_dim, config_idx):
    path = f"{SAVE_DIR}/ae_config_{config_idx}_f1{f1:.3f}.pt"
    torch.save({
        "model_state_dict" : model.state_dict(),
        "config"           : cfg,
        "f1"               : f1,
        "threshold"        : threshold,
        "input_dim"        : input_dim,
    }, path)
    return path


# ---------------------Configurations -----------------------------------

input_dim = 31

configs = [
    {"hidden": 16, "bottleneck": 2, "lr": 1e-3, "batch_size": 2048, "epochs": 300},
    {"hidden": 32, "bottleneck": 2, "lr": 1e-3, "batch_size": 2048, "epochs": 300},
    {"hidden": 64, "bottleneck": 2, "lr": 1e-3, "batch_size": 2048, "epochs": 300},
    {"hidden": 32, "bottleneck": 2, "lr": 5e-4, "batch_size": 2048, "epochs": 300},
    {"hidden": 16, "bottleneck": 2, "lr": 2e-4, "batch_size": 2048, "epochs": 300},
]



# --------------------Pre-Computation-----------------------------


print("\nMoving data to GPU")
X_tensor_legit_gpu = torch.tensor(X_legit_ae, dtype=torch.float32).to(device)
X_tensor_all       = torch.tensor(X_all_ae,   dtype=torch.float32).to(device)
print(f"X_legit : {X_tensor_legit_gpu.shape}")
print(f"X_all   : {X_tensor_all.shape}")
print(f"VRAM    : {(torch.cuda.mem_get_info()[1]-torch.cuda.mem_get_info()[0])/1e9:.1f} GB used")

#------------------------- TUNING LOOP--------------------------------------

results          = []
best_f1          = -1
best_cfg         = None
best_model_state = None
best_threshold   = None

print(f"\n{'#':<4} {'Hidden':<8} {'BN':<5} {'LR':<8} {'EP':<5} {'F1':<10} {'Time(s)'}")
print("-" * 50)

for i, cfg in enumerate(configs):
    print(f"\nConfig {i+1}/{len(configs)} — "
          f"hidden={cfg['hidden']} bn={cfg['bottleneck']} "
          f"lr={cfg['lr']} ep={cfg['epochs']}")

    t0        = time.time()
    model     = build_ae(input_dim, cfg["hidden"], cfg["bottleneck"]).to(device)
    model     = torch.compile(model)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg["lr"])
    loss_fn   = nn.MSELoss()
    scaler    = GradScaler("cuda")
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=cfg["epochs"], eta_min=1e-6)

    train_ae(model, X_tensor_legit_gpu, optimizer, loss_fn,
             cfg["epochs"], scaler, cfg["batch_size"], scheduler)

    f1, threshold, _ = quick_eval(model, X_tensor_all, y_all)
    elapsed          = time.time() - t0

    # ----------- Fix: strip _orig_mod prefix from compiled model----------------------
    if f1 > best_f1:
        best_f1          = f1
        best_cfg         = cfg
        best_threshold   = threshold
        best_model_state = {
            k.replace("_orig_mod.", ""): v.clone()
            for k, v in model.state_dict().items()
        }

    results.append({**cfg, "f1": f1, "time": elapsed})

    print(f"{i+1:<4} {cfg['hidden']:<8} {cfg['bottleneck']:<5} {cfg['lr']:<8} "
          f"{cfg['epochs']:<5} {f1:<10.4f} {elapsed:.2f}s")


#---------------------------- TUNING SUMMARY--------------------------

print("\nHYPERPARAMETER TUNING SUMMARY")
print(f"{'#':<4} {'Hidden':<8} {'BN':<5} {'LR':<8} {'EP':<5} {'F1'}")
print("-" * 40)
for i, r in enumerate(results):
    print(f"{i+1:<4} {r['hidden']:<8} {r['bottleneck']:<5} "
          f"{r['lr']:<8} {r['epochs']:<5} {r['f1']:.4f}")

print(f"\nBest Config - Hidden: {best_cfg['hidden']}  "
      f"Bottleneck: {best_cfg['bottleneck']}  "
      f"LR: {best_cfg['lr']}  Epochs: {best_cfg['epochs']}")


#--------------------------- Performance Metrice for best model----------------------------

best_model = build_ae(
    input_dim,
    best_cfg["hidden"],
    best_cfg["bottleneck"]
).to(device)
best_model.load_state_dict(best_model_state)

full_metrics(best_model, X_tensor_all, y_all, best_threshold)

# Save
torch.save({
    "model_state_dict" : best_model_state,
    "config"           : best_cfg,
    "f1"               : best_f1,
    "threshold"        : best_threshold,
    "input_dim"        : input_dim,
}, f"{SAVE_DIR}/best_model.pt")
print(f"\nBest model saved - {SAVE_DIR}/best_model.pt")

NaN in X_legit_ae : 1170629
NaN in X_all_ae   : 1679808
NaN/inf cleaned

Moving data to GPU
X_legit : torch.Size([1289852, 31])
X_all   : torch.Size([1852394, 31])
VRAM    : 0.8 GB used

#    Hidden   BN    LR       EP    F1         Time(s)
--------------------------------------------------

Config 1/5 — hidden=16 bn=2 lr=0.001 ep=300
    Epoch [50/300]  Loss: 0.552137  LR: 0.0009331
    Epoch [100/300]  Loss: 0.524796  LR: 0.0007503
    Epoch [150/300]  Loss: 0.521358  LR: 0.0005005
    Epoch [200/300]  Loss: 0.519671  LR: 0.0002507
    Epoch [250/300]  Loss: 0.518730  LR: 0.0000679
    Epoch [300/300]  Loss: 0.518415  LR: 0.0000010
1    16       2     0.001    300   0.1409     483.28s

Config 2/5 — hidden=32 bn=2 lr=0.001 ep=300
    Epoch [50/300]  Loss: 0.508036  LR: 0.0009331
    Epoch [100/300]  Loss: 0.499069  LR: 0.0007503
    Epoch [150/300]  Loss: 0.493607  LR: 0.0005005
    Epoch [200/300]  Loss: 0.489862  LR: 0.0002507
    Epoch [250/300]  Loss: 0.487256  LR: 0.0000679
    E

In [6]:
import numpy as np
import torch
import xgboost as xgb
from sklearn.metrics import classification_report, roc_auc_score
import pickle

# ---------------------------- Load data -----------------------------------------------------
X_train_xgb = np.load("/content/fraud_data/X_train_xgb.npy")
X_test_xgb  = np.load("/content/fraud_data/X_test_xgb.npy")
y_train     = np.load("/content/fraud_data/y_train.npy")
y_test      = np.load("/content/fraud_data/y_test.npy")

# Clean NaN
X_train_xgb = np.nan_to_num(X_train_xgb, nan=0.0, posinf=0.0, neginf=0.0)
X_test_xgb  = np.nan_to_num(X_test_xgb,  nan=0.0, posinf=0.0, neginf=0.0)

# -------------------------- Load best AE + generate errors ---------------------------------------
checkpoint = torch.load(
    "/content/fraud_data/models/best_model.pt",
    weights_only=False
)
ae_model = build_ae(
    checkpoint["input_dim"],
    checkpoint["config"]["hidden"],
    checkpoint["config"]["bottleneck"]
).to(device)
ae_model.load_state_dict(checkpoint["model_state_dict"])
ae_model.eval()

def get_ae_errors(model, X, device, batch_size=4096):
    errors   = []
    X_tensor = torch.tensor(
        np.nan_to_num(X, nan=0.0), dtype=torch.float32
    ).to(device)
    with torch.no_grad():
        for start in range(0, len(X), batch_size):
            batch = X_tensor[start:start + batch_size]
            recon = model(batch)
            err   = torch.mean((batch - recon) ** 2, dim=1)
            errors.append(err.cpu().numpy())
    return np.concatenate(errors)

print("Generating AE errors")
ae_errors_train = get_ae_errors(ae_model, X_train_xgb, device)
ae_errors_test  = get_ae_errors(ae_model, X_test_xgb,  device)

# ------------------- Augment features ---------------------------------
X_train_aug = np.column_stack([X_train_xgb, ae_errors_train])
X_test_aug  = np.column_stack([X_test_xgb,  ae_errors_test])
print(f"Features : {X_train_aug.shape[1]}  (31 + AE error)")

# -------------------------- Train XGBoost ----------------------------------------
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight : {scale_pos_weight:.1f}")

model_xgb = xgb.XGBClassifier(
    n_estimators          = 1000,
    max_depth             = 6,
    learning_rate         = 0.05,
    scale_pos_weight      = scale_pos_weight,
    subsample             = 0.8,
    colsample_bytree      = 0.8,
    eval_metric           = "aucpr",
    device                = "cuda",
    random_state          = 42,
    early_stopping_rounds = 50,
)

print("\nTraining XGBoost")
model_xgb.fit(
    X_train_aug, y_train,
    eval_set = [(X_test_aug, y_test)],
    verbose  = 100,
)

# -------------------------- Evaluate ---------------------------------
preds      = model_xgb.predict(X_test_aug)
preds_prob = model_xgb.predict_proba(X_test_aug)[:, 1]

print("XGBOOST - PERFORMANCE METRICS")
print(classification_report(
    y_test, preds,
    target_names=["Legit", "Fraud"],
    digits=4
))
print(f"ROC-AUC : {roc_auc_score(y_test, preds_prob):.4f}")

# ------------------------------------ Save ------------------------------------------------
with open("/content/fraud_data/models/xgb_model.pkl", "wb") as f:
    pickle.dump(model_xgb, f)
print("XGBoost saved")

Generating AE errors
Features : 32  (31 + AE error)
scale_pos_weight : 189.0

Training XGBoost
[0]	validation_0-aucpr:0.43824
[100]	validation_0-aucpr:0.93850
[200]	validation_0-aucpr:0.96036
[300]	validation_0-aucpr:0.96650
[400]	validation_0-aucpr:0.96990
[500]	validation_0-aucpr:0.97179
[600]	validation_0-aucpr:0.97300
[700]	validation_0-aucpr:0.97350
[800]	validation_0-aucpr:0.97383
[882]	validation_0-aucpr:0.97409
XGBOOST - PERFORMANCE METRICS
              precision    recall  f1-score   support

       Legit     0.9997    0.9994    0.9996    552891
       Fraud     0.8921    0.9473    0.9189      2828

    accuracy                         0.9991    555719
   macro avg     0.9459    0.9734    0.9592    555719
weighted avg     0.9992    0.9991    0.9992    555719

ROC-AUC : 0.9991
XGBoost saved


In [7]:
!pip install llama-index-llms-groq groq -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 106.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 87.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 70.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 136.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.4/142.4 kB 16.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>

In [8]:
# ------- Load context ---------------------------------------------
df_context      = pd.read_csv("/content/fraud_data/transaction_context.csv")
df_context["cc_num"] = df_context["cc_num"].astype(str)
df_context_test = df_context.iloc[len(y_train):].reset_index(drop=True)

print(f"Test context rows  : {len(df_context_test)}")
print(f"Flagged by XGBoost : {preds.sum():,}")

Test context rows  : 555719
Flagged by XGBoost : 3,003


In [9]:
#----------TOOLS-----------------------------------
def get_transaction_details(trans_num: str) -> str:
    """Fetch full details of a transaction."""
    row = df_context_test[df_context_test["trans_num"] == trans_num]
    if row.empty:
        return "Transaction not found."
    r = row.iloc[0]
    return f"""
    Amount   : ${float(r['amt']):.2f}
    Merchant : {r['merchant']}
    Category : {r['category']}
    Time     : {r['trans_date_trans_time']}
    Location : {r['city']}, {r['state']}
    Gender   : {r['gender']}  Job: {r['job']}
    """

In [10]:
def analyze_behavior(trans_num: str) -> str:
    """Analyze cardholder spending vs historical average."""
    row = df_context_test[df_context_test["trans_num"] == trans_num]
    if row.empty:
        return "Transaction not found."
    r       = row.iloc[0]
    amt     = float(r["amt"])
    history = df_context[df_context["cc_num"] == str(r["cc_num"])]
    avg     = history["amt"].mean()
    ratio   = amt / (avg + 1e-6)
    return f"""
    Current amount : ${amt:.2f}
    Historical avg : ${avg:.2f}
    Ratio          : {ratio:.2f}x
    Unusual        : {'YES' if ratio > 3 else 'NO'}
    """

In [11]:
def analyze_location(trans_num: str) -> str:
    """Check distance between cardholder home and merchant."""
    row = df_context_test[df_context_test["trans_num"] == trans_num]
    if row.empty:
        return "Transaction not found."
    r  = row.iloc[0]
    lat1, lon1 = float(r["lat"]),       float(r["long"])
    lat2, lon2 = float(r["merch_lat"]), float(r["merch_long"])
    R  = 3958.8
    a  = np.sin(np.radians((lat2-lat1)/2))**2 + \
         np.cos(np.radians(lat1))*np.cos(np.radians(lat2))* \
         np.sin(np.radians((lon2-lon1)/2))**2
    dist = R * 2 * np.arcsin(np.sqrt(a))
    return f"""
    Distance from home : {dist:.1f} miles
    High risk          : {'YES' if dist > 100 else 'NO'}
    """

In [12]:
def analyze_time(trans_num: str) -> str:
    """Check for suspicious transaction timing."""
    row = df_context_test[df_context_test["trans_num"] == trans_num]
    if row.empty:
        return "Transaction not found."
    dt   = pd.to_datetime(row.iloc[0]["trans_date_trans_time"])
    hour = dt.hour
    return f"""
    Time      : {dt.strftime('%Y-%m-%d %H:%M')}
    Hour      : {hour}:00
    Late night: {'YES' if 2 <= hour <= 6 else 'NO'}
    Weekend   : {'YES' if dt.weekday() >= 5 else 'NO'}"""

In [14]:
!pip install groq -q

os.environ["GROQ_API_KEY"] = "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"
api_key = os.environ.get("GROQ_API_KEY")
if not api_key:
    raise ValueError("Please set the GROQ_API_KEY environment variable before running this cell.")

client = GroqClient(api_key=api_key)

def investigate(trans_num, ae_error, xgb_prob):
    # Run all tools manually
    details  = get_transaction_details(trans_num)
    behavior = analyze_behavior(trans_num)
    location = analyze_location(trans_num)
    timing   = analyze_time(trans_num)

    prompt = f"""
    A transaction has been flagged for fraud analysis.

    Transaction ID                     : {trans_num}
    AE Reconstruction Error            : {ae_error:.6f}
    XGBoost Fraud Probability (0 to 1) : {xgb_prob:.3f}
    Investigation Results:
    TRANSACTION DETAILS:{details}
    BEHAVIORAL ANALYSIS:{behavior}
    LOCATION ANALYSIS:{location}
    TIME ANALYSIS:{timing}

    Important instructions:
    - Treat the AE reconstruction error as an input feature, not as a standalone threshold.
    - Use presentation-safe wording. If a merchant name contains the token 'fraud', describe it as 'suspicious naming patterns in the dataset' rather than claiming real-world certainty.
    - Keep the final assessment aligned with the recommendation.
      * FRAUD + HIGH confidence -> BLOCK
      * LEGITIMATE + HIGH confidence -> ALLOW
      * Any MEDIUM/LOW confidence case -> REVIEW

    Return the result in exactly this format:

    Final Assessment (LLM-assisted): FRAUD / LEGITIMATE
    Confidence                    : HIGH / MEDIUM / LOW
    Evidence:
    - bullet point 1
    - bullet point 2
    Recommendation               : BLOCK / ALLOW / REVIEW
    """

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
    )
    return response.choices[0].message.content



In [15]:
import time
import re

def investigate_with_retry(trans_num, ae_error, xgb_prob, max_retries=3):
    for attempt in range(max_retries):
        try:
            return investigate(trans_num, ae_error, xgb_prob)
        except Exception as e:
            if "429" in str(e) or "TooManyRequests" in str(e):
                wait = 60 * (attempt + 1)
                print(f"Rate limit hit — waiting {wait}s before retry {attempt+1}/{max_retries}")
                time.sleep(wait)
            else:
                raise e
    return "INVESTIGATION FAILED — rate limit exceeded"


def extract_field(report, field_name):
    pattern = rf"{re.escape(field_name)}\s*:\s*(.+)"
    match = re.search(pattern, report, flags=re.IGNORECASE)
    return match.group(1).strip() if match else "UNKNOWN"


# ----------------------- Run with retry + delay----------------------------------------------------
fraud_idx = np.where((preds == 1) & (y_test == 1))[0][:3]
legit_idx = np.where((preds == 0) & (y_test == 0))[0][:2]

results = []

for idx in np.concatenate([fraud_idx, legit_idx]):
    trans_num = df_context_test.iloc[idx]["trans_num"]
    true_label = "FRAUD" if y_test[idx] == 1 else "LEGITIMATE"
    ae_error = float(ae_errors_test[idx])
    xgb_prob = float(preds_prob[idx])


    print(f"Transaction ID              : {trans_num}")
    print(f"True Label                  : {true_label}")
    print(f"AE Error                    : {ae_error:.6f}")
    print(f"XGBoost Probability (0 to 1): {xgb_prob:.3f}")

    report = investigate_with_retry(trans_num, ae_error, xgb_prob)
    print(report)

    final_assessment = extract_field(report, "Final Assessment (LLM-assisted)").upper()
    confidence = extract_field(report, "Confidence").upper()
    recommendation = extract_field(report, "Recommendation").upper()

    if "LEGITIMATE" in final_assessment:
        llm_decision = "LEGITIMATE"
    elif "FRAUD" in final_assessment:
        llm_decision = "FRAUD"
    else:
        llm_decision = "UNKNOWN"

    is_correct = llm_decision == true_label
    print(f"Ground Truth Match          : {' Correct' if is_correct else 'Incorrect'}")

    results.append({
        "trans_num": trans_num,
        "true_label": true_label,
        "llm_decision": llm_decision,
        "confidence": confidence,
        "recommendation": recommendation,
        "correct": is_correct,
        "ae_error": ae_error,
        "xgb_prob": xgb_prob,
    })

print("All investigations complete")

# ----------------------------- Final metrics --------------------------------------
df_results = pd.DataFrame(results)
print("LLM REASONING LAYER - RESULTS")
print(df_results[["trans_num", "true_label", "llm_decision", "confidence", "recommendation", "correct", "xgb_prob"]].to_string(index=False))
print(f"Accuracy : {df_results['correct'].mean()*100:.1f}%  ({df_results['correct'].sum()}/{len(df_results)} correct)")



Transaction ID              : 8eaa38d778d7123f997477d251fd6196
True Label                  : FRAUD
AE Error                    : 0.910132
XGBoost Probability (0 to 1): 0.990
Final Assessment (LLM-assisted): FRAUD
Confidence                    : HIGH
Evidence:
- The XGBoost Fraud Probability is extremely high at 0.990, indicating a strong likelihood of fraudulent activity.
- The transaction amount of $838.98 is significantly higher than the historical average of $60.85, with a ratio of 13.79x, which is unusual and raises suspicions.
- The merchant name exhibits suspicious naming patterns in the dataset, which may suggest potential fraudulent intent.
Recommendation               : BLOCK
Ground Truth Match          :  Correct
Transaction ID              : 721662305df96a1e25ece300966489ae
True Label                  : FRAUD
AE Error                    : 1.194780
XGBoost Probability (0 to 1): 1.000
Final Assessment (LLM-assisted): FRAUD
Confidence                    : HIGH
Evidence:
- The X